# Stage 3 Qwen3-4B Prompt Repair Clean (Kaggle)

Targeted prompt-repair sweep for `Qwen/Qwen3-VL-4B-Instruct` under the clean Stage 3 GT-crop protocol.

The model sweep showed that Qwen3-4B improves raw `insulator_ok` recognition but severely undercalls `defect_flashover`. This notebook keeps model, dataset, schema, and evaluator fixed, and tests three small prompt variants that try to recover flashover recall.


In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display

REPO_URL = "https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git"
REPO_DIR = Path("/kaggle/working/vlm-for-insulator-defect-detection")

DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"),
    Path("/kaggle/input/idid-coco-v3"),
]
JSONL_REL = Path("stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl")

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
BACKEND_MODE = "qwen_hf"
RUN_ID_PREFIX = "stage3_qwen3_4b_prompt_repair_clean"
MAX_PIXELS = 401408

PROMPT_VERSIONS = [
    "qwen_vlm_labels_v1_prompt_v7f_flashover_unclear_to_unknown_nocroppath",
    "qwen_vlm_labels_v1_prompt_v8a_qwen3_flashover_recall_nocroppath",
    "qwen_vlm_labels_v1_prompt_v8b_qwen3_flashover_balanced_nocroppath",
    "qwen_vlm_labels_v1_prompt_v8c_qwen3_flashover_positive_evidence_nocroppath",
]
CONTROL_VERSION = "qwen_vlm_labels_v1_prompt_v7f_flashover_unclear_to_unknown_nocroppath"

BASELINE_3B_CORRECT = 28
BASELINE_3B_MACRO_F1 = 0.2946488294314381
BASELINE_3B_FLASHOVER_RECALL = 0.75

print("Model:", MODEL_ID)
print("Prompt versions:", len(PROMPT_VERSIONS))


In [ ]:
def sh(args, cwd: Path | None = None, check: bool = True):
    print("$", " ".join(str(a) for a in args))
    p = subprocess.run(
        [str(a) for a in args],
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
    )
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(str(a) for a in args)}")
    return p

def require_path(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    return path

def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


In [ ]:
gpu = sh(["nvidia-smi"], check=False)
if gpu.returncode != 0:
    raise RuntimeError("GPU is required for this prompt repair sweep")
sh(["python", "-c", "import torch; print('cuda_available=', torch.cuda.is_available()); print('device_count=', torch.cuda.device_count())"], check=True)


In [ ]:
DATA_ROOT = None
for root in DATASET_ROOT_CANDIDATES:
    if (root / JSONL_REL).exists():
        DATA_ROOT = root
        break
if DATA_ROOT is None:
    raise FileNotFoundError("Could not find clean Stage 3 JSONL in Kaggle inputs")
input_jsonl = require_path(DATA_ROOT / JSONL_REL, "clean val jsonl")
print("DATA_ROOT:", DATA_ROOT)
print("input_jsonl:", input_jsonl)


In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
sh(["git", "clone", REPO_URL, str(REPO_DIR)])

sh([
    "python", "-m", "pip", "install", "-q", "-U",
    "transformers>=4.57.0", "accelerate", "qwen-vl-utils", "pillow", "pandas", "pyyaml", "tabulate",
], cwd=REPO_DIR)
print("Repo ready:", REPO_DIR)


In [ ]:
base_cfg_path = REPO_DIR / "configs/pipeline/stage3_vlm_gt_baseline.yaml"
base_cfg = yaml.safe_load(base_cfg_path.read_text(encoding="utf-8"))

missing = []
versions = base_cfg.get("prompt", {}).get("versions", {})
for pv in PROMPT_VERSIONS:
    if pv not in versions:
        missing.append(pv)
if missing:
    raise KeyError("Missing prompt versions: " + ", ".join(missing))

required_patterns = {
    "qwen_vlm_labels_v1_prompt_v8a_qwen3_flashover_recall_nocroppath": "prefer defect_flashover over insulator_ok",
    "qwen_vlm_labels_v1_prompt_v8b_qwen3_flashover_balanced_nocroppath": "Qwen3 tendency guard",
    "qwen_vlm_labels_v1_prompt_v8c_qwen3_flashover_positive_evidence_nocroppath": "positive flashover evidence",
}
for pv, pattern in required_patterns.items():
    sys_path = REPO_DIR / versions[pv]["system_path"]
    text = sys_path.read_text(encoding="utf-8")
    if pattern not in text:
        raise AssertionError(f"Prompt check failed for {pv}: missing {pattern}")
print("Prompt preflight OK")


def make_config(prompt_version: str) -> Path:
    cfg = json.loads(json.dumps(base_cfg))
    cfg["input"]["dataset_jsonl"] = str(input_jsonl)
    cfg["backend"]["mode"] = BACKEND_MODE
    cfg["prompt"]["version"] = prompt_version
    qwen_cfg = cfg["backend"].setdefault("qwen_hf", {})
    qwen_cfg["model_id"] = MODEL_ID
    qwen_cfg["max_pixels"] = MAX_PIXELS
    qwen_cfg["torch_dtype"] = "auto"
    qwen_cfg["device_map"] = "auto"
    qwen_cfg["trust_remote_code"] = True
    cfg["run"]["resume"] = False
    short = prompt_version.replace("qwen_vlm_labels_v1_prompt_", "")
    cfg_path = REPO_DIR / "configs" / f"stage3_qwen3_4b_prompt_repair_{short}.yaml"
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=False), encoding="utf-8")
    return cfg_path

def read_confusion_counts(eval_dir: Path) -> dict:
    path = eval_dir / "confusion_coarse_class.csv"
    if not path.exists():
        return {}
    df = pd.read_csv(path)
    out = {}
    gt_col = "gt\\pred"
    for _, row in df.iterrows():
        gt = row[gt_col]
        total = 0
        correct = 0
        for col in df.columns:
            if col == gt_col:
                continue
            value = int(row[col])
            total += value
            if col == gt:
                correct += value
        out[f"{gt}_total"] = total
        out[f"{gt}_correct"] = correct
        out[f"{gt}_recall"] = (correct / total) if total else None
    return out

def metrics_row(prompt_version: str, run_id: str, preflight_ok: bool, full_ok: bool, error: str | None):
    run_dir = REPO_DIR / "outputs/stage3_vlm_baseline_runs" / run_id
    eval_dir = run_dir / "eval"
    row = {"prompt_version": prompt_version, "run_id": run_id, "preflight_ok": preflight_ok, "full_ok": full_ok, "error": error or ""}
    metrics_path = eval_dir / "metrics.json"
    if metrics_path.exists():
        m = read_json(metrics_path)
        rates = m.get("rates", {})
        f1 = m.get("f1", {})
        counts = m.get("counts", {})
        evaluated = counts.get("evaluated_total")
        coarse_acc = rates.get("coarse_class_accuracy")
        coarse_correct_total = round(coarse_acc * evaluated) if isinstance(evaluated, int) and isinstance(coarse_acc, (int, float)) else None
        row.update({
            "evaluated_total": evaluated,
            "parse_success": rates.get("parse_success_rate"),
            "schema_valid": rates.get("schema_valid_rate"),
            "coarse_acc": coarse_acc,
            "coarse_correct_total": coarse_correct_total,
            "coarse_macro_f1": f1.get("coarse_class_macro_f1"),
            "visibility_acc": rates.get("visibility_accuracy"),
            "visibility_macro_f1": f1.get("visibility_macro_f1"),
            "needs_review_acc": rates.get("needs_review_accuracy"),
            "tag_mean_jaccard": rates.get("tag_mean_jaccard"),
            "pred_ambiguous_rate": rates.get("pred_ambiguous_rate"),
        })
        row.update(read_confusion_counts(eval_dir))
    return row


In [ ]:
rows = []
completed_run_ids = []
for pv in PROMPT_VERSIONS:
    short = pv.replace("qwen_vlm_labels_v1_prompt_", "")
    cfg_path = make_config(pv)
    preflight_id = f"{RUN_ID_PREFIX}_{short}_preflight"
    full_id = f"{RUN_ID_PREFIX}_{short}"
    preflight_ok = False
    full_ok = False
    error = None

    try:
        sh([
            "python", "scripts/run_stage3_vlm_baseline.py",
            "--config", str(cfg_path),
            "--run-id", preflight_id,
            "--max-samples", "1",
            "--no-resume",
        ], cwd=REPO_DIR)
        preflight_ok = True
    except Exception as exc:
        error = f"preflight failed: {exc}"
        print(error)

    if preflight_ok:
        try:
            run_dir = REPO_DIR / "outputs/stage3_vlm_baseline_runs" / full_id
            sh([
                "python", "scripts/run_stage3_vlm_baseline.py",
                "--config", str(cfg_path),
                "--run-id", full_id,
                "--no-resume",
            ], cwd=REPO_DIR)
            sh(["python", "scripts/validate_vlm_labels_v1.py", "--input", str(run_dir / "predictions_vlm_labels_v1.jsonl")], cwd=REPO_DIR)
            sh([
                "python", "scripts/eval_stage3_vlm_baseline.py",
                "--run-dir", str(run_dir),
                "--ground-truth-jsonl", str(input_jsonl),
                "--output-dir", str(run_dir / "eval"),
            ], cwd=REPO_DIR)
            full_ok = True
            completed_run_ids.append(full_id)
        except Exception as exc:
            error = f"full run failed: {exc}"
            print(error)

    rows.append(metrics_row(pv, full_id, preflight_ok, full_ok, error))

comparison = pd.DataFrame(rows)
display(comparison)


In [ ]:
def verdict_table(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    verdicts = []
    for _, row in out.iterrows():
        if not bool(row.get("full_ok")):
            verdicts.append("NOT_RUN")
            continue
        if row.get("prompt_version") == CONTROL_VERSION:
            verdicts.append("CONTROL")
            continue
        parse_ok = row.get("parse_success") == 1.0 and row.get("schema_valid") == 1.0
        flashover_recall = row.get("defect_flashover_recall")
        ok_recall = row.get("insulator_ok_recall")
        macro_f1 = row.get("coarse_macro_f1")
        correct = row.get("coarse_correct_total")
        if parse_ok and pd.notna(flashover_recall) and pd.notna(ok_recall) and pd.notna(macro_f1) and pd.notna(correct):
            if flashover_recall >= 0.50 and ok_recall >= 0.50 and macro_f1 >= BASELINE_3B_MACRO_F1 and correct >= BASELINE_3B_CORRECT:
                verdicts.append("STRONG_KEEP")
            elif flashover_recall > 0.10 and ok_recall >= 0.50:
                verdicts.append("WEAK_REPAIR")
            else:
                verdicts.append("NO_REPAIR")
        else:
            verdicts.append("NO_REPAIR")
    out["verdict"] = verdicts
    if "coarse_macro_f1" in out.columns:
        out = out.sort_values(["full_ok", "coarse_macro_f1", "coarse_acc", "defect_flashover_recall"], ascending=[False, False, False, False])
    return out

comparison = verdict_table(comparison)
display(comparison)


In [ ]:
out_dir = REPO_DIR / "outputs/stage3_qwen3_4b_prompt_repair_clean"
out_dir.mkdir(parents=True, exist_ok=True)
comparison.to_csv(out_dir / "prompt_repair_comparison.csv", index=False)
md_lines = [
    "# Stage 3 Qwen3-4B Prompt Repair Clean",
    "",
    f"Model: `{MODEL_ID}`",
    f"Max pixels: `{MAX_PIXELS}`",
    "",
    comparison.to_markdown(index=False),
]
(out_dir / "prompt_repair_comparison.md").write_text("\n".join(md_lines), encoding="utf-8")

completed = comparison[comparison["full_ok"] == True].copy()
if not completed.empty:
    ranked = completed.sort_values(["coarse_macro_f1", "coarse_acc", "defect_flashover_recall"], ascending=[False, False, False])
    best = ranked.iloc[0].to_dict()
    print("Best prompt:", best.get("prompt_version"))
    print("coarse_acc:", best.get("coarse_acc"), "macro_f1:", best.get("coarse_macro_f1"), "flashover_recall:", best.get("defect_flashover_recall"))
    print("verdict:", best.get("verdict"))
else:
    print("No full prompt run completed.")

package_root = REPO_DIR / "outputs/stage3_qwen3_4b_prompt_repair_package"
if package_root.exists():
    shutil.rmtree(package_root)
package_root.mkdir(parents=True, exist_ok=True)
shutil.copy2(out_dir / "prompt_repair_comparison.csv", package_root / "prompt_repair_comparison.csv")
shutil.copy2(out_dir / "prompt_repair_comparison.md", package_root / "prompt_repair_comparison.md")

runs_out = package_root / "stage3_vlm_baseline_runs"
runs_out.mkdir(parents=True, exist_ok=True)
for run_id in completed_run_ids:
    src = REPO_DIR / "outputs/stage3_vlm_baseline_runs" / run_id
    if src.exists():
        shutil.copytree(src, runs_out / run_id)

archive_path = Path("/kaggle/working/stage3_deliverables_qwen3_4b_prompt_repair_clean.tar.gz")
if archive_path.exists():
    archive_path.unlink()
sh(["tar", "-czf", str(archive_path), "-C", str(package_root.parent), package_root.name], check=True)
print("Archive:", archive_path)
